In [ ]:
# Imports and environment first, as always

import os
import random
import requests
from typing import Annotated, Literal
from typing_extensions import TypedDict
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langchain_openai import ChatOpenAI
from langchain_community.tools import GoogleSerperRun
from langchain_community.utilities import GoogleSerperAPIWrapper
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from langgraph.checkpoint.sqlite import SqliteSaver
import telebot
from datetime import datetime
from zoneinfo import ZoneInfo


load_dotenv(override=True)

bot = telebot.TeleBot(os.environ.get("TELEGRAM_BOT_TOKEN"))


### Custom calculator tool for the agent


In [ ]:
@tool
def calculator(
    a: float, b: float, operation: Literal["add", "subtract", "multiply", "divide"]
) -> float:
    """Perform basic arithmetic on two numbers."""

    if operation == "add":
        return a + b

    elif operation == "subtract":
        return a - b

    elif operation == "multiply":
        return a * b

    elif operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b


# Literal means the value must one of the pre-defined values in the list.


@tool
def get_today_date() -> str:
    """Get today's date in YYYY-MM-DD format."""

    return datetime.now(tz=ZoneInfo("Asia/Singapore")).strftime("%Y-%m-%d")

In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]
    chinese: str


llm = ChatOpenAI(model="gpt-5.4-mini")

In [ ]:
search = GoogleSerperRun(api_wrapper=GoogleSerperAPIWrapper())


@tool
def send_push_notification(text: str) -> str:
    """Send a short push notification to the user's phone."""
    bot.send_message(chat_id=os.getenv("TELEGRAM_USER_ID"), text=text)
    return "Notification sent"


tools = [search, send_push_notification, calculator]
llm_with_tools = llm.bind_tools(tools)  # this is a functionality we saw in day 1


In [ ]:
def translator_node(state: State) -> dict:
    last = state["messages"][-1].content
    prompt = f"Translate this into simplified chinese, replying with the translation only:\n\n{last}"
    return {"chinese": llm.invoke(prompt).content}


def chatbot_node(state: State) -> dict:
    return {
        "messages": [llm_with_tools.invoke(state["messages"])]
    }  # it makes a call with the LLM with tools bound to it


In [ ]:
memory = MemorySaver()  # this is a checkpointer

builder = StateGraph(State)
builder.add_node("chatbot", chatbot_node)
builder.add_node("tools", ToolNode(tools))
builder.add_node("translator", translator_node)
builder.add_edge(START, "chatbot")
builder.add_conditional_edges(
    "chatbot", tools_condition, {"tools": "tools", END: "translator"}
)
builder.add_edge("translator", END)
builder.add_edge("tools", "chatbot")
graph = builder.compile(checkpointer=memory)


In [ ]:
def chat(user_input: str, history):
    config = {"configurable": {"thread_id": "gradio-session4"}}
    result = graph.invoke(
        {"messages": [{"role": "user", "content": user_input}]}, config
    )
    return f"{result['messages'][-1].content}\n\n*{result['chinese']}*"


gr.ChatInterface(chat).launch()
